In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')



**Load Dataset**

In [4]:
train_df = pd.read_csv('train_noisy.csv')
test_df = pd.read_csv('test_noisy.csv')

In [6]:
train_df.head()

,ID,Query,Priority
0,1,"Hello sir, How to book auditorium",Low
1,2,Parking rules on campus please respond soon.,Low
2,3,"Respected sir/madam, Query about attendance sh...",High
3,4,"Hello sir, Exam center not allotted yet asap p...",High
4,5,"Hello sir, Information about semester break da...",Low


In [7]:
train_df.columns

Index(['ID', 'Query', 'Priority'], dtype='object')

In [8]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   ID        5000 non-null   int64 
 1   Query     5000 non-null   object
 2   Priority  5000 non-null   object
dtypes: int64(1), object(2)
memory usage: 117.3+ KB


In [11]:
train_df['Priority'].value_counts()

,count
Priority,
Low,1717
High,1655
Medium,1628


#Preprocessing
clean text and encode labels


In [12]:
import re
import string

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["clean_query"] = train_df["Query"].apply(clean_text)
test_df["clean_query"] = test_df["Query"].apply(clean_text)

In [13]:
train_df.head()

,ID,Query,Priority,clean_query
0,1,"Hello sir, How to book auditorium",Low,hello sir how to book auditorium
1,2,Parking rules on campus please respond soon.,Low,parking rules on campus please respond soon
2,3,"Respected sir/madam, Query about attendance sh...",High,respected sirmadam query about attendance shor...
3,4,"Hello sir, Exam center not allotted yet asap p...",High,hello sir exam center not allotted yet asap pl...
4,5,"Hello sir, Information about semester break da...",Low,hello sir information about semester break dat...


In [14]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

In [15]:
train_df["priority_encoded"] = le.fit_transform(train_df["Priority"])
test_df["priority_encoded"] = le.transform(test_df["Priority"])
print("Classes:", list(le.classes_))

Classes: ['High', 'Low', 'Medium']


In [16]:
train_df.head(10)

,ID,Query,Priority,clean_query,priority_encoded
0,1,"Hello sir, How to book auditorium",Low,hello sir how to book auditorium,1
1,2,Parking rules on campus please respond soon.,Low,parking rules on campus please respond soon,1
2,3,"Respected sir/madam, Query about attendance sh...",High,respected sirmadam query about attendance shor...,0
3,4,"Hello sir, Exam center not allotted yet asap p...",High,hello sir exam center not allotted yet asap pl...,0
4,5,"Hello sir, Information about semester break da...",Low,hello sir information about semester break dat...,1
5,6,"Hi, Urgent correction in marksheet needed",High,hi urgent correction in marksheet needed,0
6,7,"Hello sir, Campus WiFi password reset process ...",High,hello sir campus wifi password reset process p...,0
7,8,"Respected sir/madam, Urgent correction in mark...",High,respected sirmadam urgent correction in marksh...,0
8,9,"Can someone please help, When will timetable b...",Medium,can someone please help when will timetable be...,2
9,10,"Just wanted to ask, Need to change elective su...",Low,just wanted to ask need to change elective sub...,1


# Convert text to numbers
TF-IDF is used to provide numbers to each word inn the query box


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", max_features=3000)
X_train = vectorizer.fit_transform(train_df["clean_query"])
X_test = vectorizer.transform(test_df["clean_query"])

y_train = train_df["priority_encoded"]
y_test = test_df["priority_encoded"]

In [20]:
train_df.head()

,ID,Query,Priority,clean_query,priority_encoded
0,1,"Hello sir, How to book auditorium",Low,hello sir how to book auditorium,1
1,2,Parking rules on campus please respond soon.,Low,parking rules on campus please respond soon,1
2,3,"Respected sir/madam, Query about attendance sh...",High,respected sirmadam query about attendance shor...,0
3,4,"Hello sir, Exam center not allotted yet asap p...",High,hello sir exam center not allotted yet asap pl...,0
4,5,"Hello sir, Information about semester break da...",Low,hello sir information about semester break dat...,1


# Model 1 - Logistic Regression

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

params_lr = {
    "C": [0.1, 1, 10],
    "solver": ["lbfgs", "liblinear"]
}

grid_lr = GridSearchCV(LogisticRegression(max_iter=1000), params_lr, cv=5, scoring="accuracy")
grid_lr.fit(X_train, y_train)

print("Best params:", grid_lr.best_params_)
print("Best CV score:", grid_lr.best_score_)

model = grid_lr.best_estimator_

Best params: {'C': 0.1, 'solver': 'lbfgs'}
Best CV score: 0.881


In [23]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000, C=0.1, solver="lbfgs")


model.fit(X_train, y_train)

LogisticRegression(C=0.1, max_iter=1000)

In [24]:
y_pred = model.predict(X_test)

In [25]:
y_pred

array([0, 0, 2, 1, 1, 2, 0, 0, 0, 0, 2, 2, 1, 0, 1, 1, 2, 2, 1, 0, 0, 0,
       0, 0, 0, 0, 1, 2, 1, 1, 2, 1, 0, 1, 1, 1, 1, 2, 1, 2, 1, 1, 0, 2,
       0, 0, 2, 0, 2, 2, 0, 1, 1, 2, 0, 2, 0, 2, 1, 0, 1, 0, 2, 1, 0, 2,
       2, 2, 1, 0, 0, 0, 2, 1, 2, 1, 1, 1, 0, 0, 2, 2, 0, 1, 2, 0, 1, 1,
       0, 1, 1, 2, 2, 2, 2, 1, 1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2, 1, 2,
       1, 0, 2, 0, 0, 2, 1, 1, 0, 2, 1, 2, 2, 1, 1, 2, 1, 2, 0, 2, 0, 1,
       1, 0, 0, 2, 2, 0, 0, 1, 1, 1, 1, 1, 2, 1, 0, 2, 0, 0, 2, 2, 1, 0,
       0, 0, 2, 1, 1, 0, 2, 1, 0, 2, 1, 1, 1, 0, 2, 2, 0, 1, 0, 0, 0, 2,
       1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 1, 1, 0, 1, 0, 2, 2, 2, 0, 2, 0, 0,
       0, 2, 0, 2, 0, 2, 2, 1, 1, 2, 2, 1, 0, 2, 1, 2, 2, 2, 0, 2, 1, 0,
       2, 2, 1, 2, 1, 0, 1, 2, 1, 0, 2, 0, 1, 0, 0, 2, 2, 0, 2, 2, 2, 0,
       2, 1, 0, 1, 1, 1, 2, 2, 2, 2, 0, 0, 1, 2, 1, 0, 1, 1, 0, 0, 2, 0,
       2, 2, 0, 2, 2, 0, 2, 1, 0, 1, 1, 2, 1, 2, 1, 0, 0, 0, 1, 2, 2, 0,
       1, 2, 1, 2, 1, 2, 0, 1, 0, 1, 2, 1, 1, 1, 1,

In [26]:
y_test

,priority_encoded
0,0
1,1
2,2
3,1
4,1
...,...
995,2
996,2
997,1
998,2


In [27]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [28]:
accuracy_score(y_test, y_pred)

0.863

In [29]:
confusion_matrix(y_test, y_pred)

array([[257,  17,  28],
       [ 22, 309,  26],
       [ 23,  21, 297]])

In [30]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.85      0.85      0.85       302
           1       0.89      0.87      0.88       357
           2       0.85      0.87      0.86       341

    accuracy                           0.86      1000
   macro avg       0.86      0.86      0.86      1000
weighted avg       0.86      0.86      0.86      1000



In [31]:
from sklearn.neighbors import KNeighborsClassifier

In [32]:
params_knn = {
    "n_neighbors": [3, 5, 7, 9],
    "weights": ["uniform", "distance"]
}

grid_knn = GridSearchCV(KNeighborsClassifier(), params_knn, cv=5, scoring="accuracy")
grid_knn.fit(X_train, y_train)

print("Best params:", grid_knn.best_params_)
print("Best CV score:", grid_knn.best_score_)

knn_model = grid_knn.best_estimator_

Best params: {'n_neighbors': 9, 'weights': 'uniform'}
Best CV score: 0.8808


In [34]:
knn_model = KNeighborsClassifier(n_neighbors=5,weights='uniform')
knn_model.fit(X_train, y_train)

KNeighborsClassifier()

In [36]:
y_pred_knn=knn_model.predict(X_test)

In [37]:
accuracy_score(y_test, y_pred_knn)

0.86

In [38]:
confusion_matrix(y_test, y_pred_knn)

array([[258,  17,  27],
       [ 25, 306,  26],
       [ 24,  21, 296]])

In [39]:
print(classification_report(y_test, y_pred_knn))

              precision    recall  f1-score   support

           0       0.84      0.85      0.85       302
           1       0.89      0.86      0.87       357
           2       0.85      0.87      0.86       341

    accuracy                           0.86      1000
   macro avg       0.86      0.86      0.86      1000
weighted avg       0.86      0.86      0.86      1000



In [40]:
from sklearn.naive_bayes import MultinomialNB

In [41]:
params_nb = {
    "alpha": [0.1, 0.5, 1.0, 2.0]
}

grid_nb = GridSearchCV(MultinomialNB(), params_nb, cv=5, scoring="accuracy")
grid_nb.fit(X_train, y_train)

print("Best params:", grid_nb.best_params_)
print("Best CV score:", grid_nb.best_score_)

model_NB = grid_nb.best_estimator_

Best params: {'alpha': 0.1}
Best CV score: 0.881


In [43]:
model_NB = MultinomialNB(alpha=0.1)
model_NB.fit(X_train, y_train)

MultinomialNB(alpha=0.1)

In [44]:
y_pred_nb=model_NB.predict(X_test)

In [45]:
y_pred_nb

array([0, 0, 2, 1, 1, 2, 0, 0, 0, 0, 2, 2, 1, 0, 1, 1, 2, 2, 1, 0, 0, 0,
       0, 0, 0, 0, 1, 2, 1, 1, 2, 1, 0, 1, 1, 1, 1, 2, 1, 2, 1, 1, 0, 2,
       0, 0, 2, 0, 2, 2, 0, 1, 1, 2, 0, 2, 0, 2, 1, 0, 1, 0, 2, 1, 0, 2,
       2, 2, 1, 0, 0, 0, 2, 1, 2, 1, 1, 1, 0, 0, 2, 2, 0, 1, 2, 0, 1, 1,
       0, 1, 1, 2, 2, 2, 2, 1, 1, 1, 2, 2, 1, 1, 2, 2, 1, 1, 2, 2, 1, 2,
       1, 0, 2, 0, 0, 2, 1, 1, 0, 2, 1, 2, 2, 1, 1, 2, 1, 2, 0, 2, 0, 1,
       1, 0, 0, 2, 2, 0, 0, 1, 1, 1, 1, 1, 2, 1, 0, 2, 0, 0, 2, 2, 1, 0,
       0, 0, 2, 1, 1, 0, 2, 1, 0, 2, 1, 1, 1, 0, 2, 2, 0, 1, 0, 0, 0, 2,
       1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 1, 1, 0, 1, 0, 2, 2, 2, 0, 2, 0, 0,
       0, 2, 0, 2, 0, 2, 2, 1, 1, 2, 2, 1, 0, 2, 1, 2, 2, 2, 0, 2, 1, 0,
       2, 2, 1, 2, 1, 0, 1, 2, 1, 0, 2, 0, 1, 0, 0, 2, 2, 0, 2, 2, 2, 0,
       2, 1, 0, 1, 1, 1, 2, 2, 2, 2, 0, 0, 1, 2, 1, 0, 1, 1, 0, 0, 2, 0,
       2, 2, 0, 2, 2, 0, 2, 1, 0, 1, 1, 2, 1, 2, 1, 0, 0, 0, 1, 2, 2, 0,
       1, 2, 1, 2, 1, 2, 0, 1, 0, 1, 2, 1, 1, 1, 1,

In [47]:
accuracy_score(y_test, y_pred_nb)

0.863

In [49]:
confusion_matrix(y_test, y_pred_nb)

array([[257,  17,  28],
       [ 22, 309,  26],
       [ 23,  21, 297]])

In [51]:
print(classification_report(y_test, y_pred_nb))

              precision    recall  f1-score   support

           0       0.85      0.85      0.85       302
           1       0.89      0.87      0.88       357
           2       0.85      0.87      0.86       341

    accuracy                           0.86      1000
   macro avg       0.86      0.86      0.86      1000
weighted avg       0.86      0.86      0.86      1000



In [52]:
from sklearn.tree import DecisionTreeClassifier

In [53]:
params_dt = {
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10]
}

grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), params_dt, cv=5, scoring="accuracy")
grid_dt.fit(X_train, y_train)

print("Best params:", grid_dt.best_params_)
print("Best CV score:", grid_dt.best_score_)

model_DT = grid_dt.best_estimator_

Best params: {'max_depth': 20, 'min_samples_split': 10}
Best CV score: 0.8337999999999999


In [54]:
model_DT = DecisionTreeClassifier(random_state=42,max_depth=10,min_samples_split=10)
model_DT.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=10, min_samples_split=10, random_state=42)

In [58]:
y_pred_DT=model_DT.predict(X_test)

In [59]:
y_pred_DT

array([0, 0, 2, 1, 1, 2, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 2, 1, 1, 0, 0, 0,
       0, 0, 0, 0, 1, 2, 1, 1, 2, 1, 0, 1, 1, 1, 1, 2, 1, 2, 1, 1, 0, 1,
       0, 0, 1, 0, 2, 1, 0, 1, 1, 2, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0,
       2, 2, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1,
       0, 1, 1, 2, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 2, 1, 1, 2, 2, 1, 1,
       1, 1, 1, 0, 0, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 2, 1, 2, 0, 1, 0, 1,
       1, 0, 0, 2, 2, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 2, 0, 1, 2, 1, 1, 0,
       0, 0, 1, 1, 1, 0, 1, 1, 0, 2, 1, 1, 1, 0, 2, 1, 0, 1, 1, 0, 1, 1,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 2, 0, 0,
       0, 1, 0, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 0,
       2, 2, 1, 1, 1, 0, 1, 2, 1, 0, 1, 0, 1, 0, 0, 2, 2, 0, 1, 1, 1, 0,
       2, 1, 0, 1, 1, 1, 1, 2, 1, 2, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0,
       1, 1, 0, 2, 2, 0, 1, 1, 1, 1, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 1, 0,
       1, 2, 1, 1, 1, 2, 0, 1, 1, 1, 2, 1, 1, 1, 1,

In [61]:
accuracy_score(y_test, y_pred_DT)

0.681

In [62]:
confusion_matrix(y_test, y_pred_DT)

array([[208,  77,  17],
       [ 19, 326,  12],
       [ 22, 172, 147]])

In [63]:
print(classification_report(y_test, y_pred_DT))

              precision    recall  f1-score   support

           0       0.84      0.69      0.75       302
           1       0.57      0.91      0.70       357
           2       0.84      0.43      0.57       341

    accuracy                           0.68      1000
   macro avg       0.75      0.68      0.67      1000
weighted avg       0.74      0.68      0.67      1000



In [64]:
from sklearn.svm import SVC

In [65]:
params_svm = {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"]
}

grid_svm = GridSearchCV(SVC(), params_svm, cv=5, scoring="accuracy")
grid_svm.fit(X_train, y_train)

print("Best params:", grid_svm.best_params_)
print("Best CV score:", grid_svm.best_score_)

model_svm = grid_svm.best_estimator_

Best params: {'C': 0.1, 'kernel': 'linear'}
Best CV score: 0.881


In [66]:
model_svm = SVC(C=0.1,kernel='linear')

In [67]:
model_svm.fit(X_train, y_train)

SVC(C=0.1, kernel='linear')

In [68]:
y_pred_svc = model_svm.predict(X_test)

In [69]:
accuracy_score(y_test, y_pred_svc)

0.863

In [70]:
confusion_matrix(y_test, y_pred_svc)

array([[257,  17,  28],
       [ 22, 309,  26],
       [ 23,  21, 297]])

In [71]:
print(classification_report(y_test, y_pred_svc))

              precision    recall  f1-score   support

           0       0.85      0.85      0.85       302
           1       0.89      0.87      0.88       357
           2       0.85      0.87      0.86       341

    accuracy                           0.86      1000
   macro avg       0.86      0.86      0.86      1000
weighted avg       0.86      0.86      0.86      1000



# Cross Validation

In [72]:
from sklearn.model_selection import cross_val_score

In [73]:
import scipy.sparse as sp

X_all = sp.vstack([X_train, X_test])
y_all = pd.concat([y_train, y_test])

In [74]:
scores = cross_val_score(model_svm, X_all, y_all, cv=5, scoring='accuracy')

In [75]:
print(scores)

[0.875      0.87833333 0.88666667 0.88166667 0.86833333]


In [76]:
print(scores.mean())

0.8780000000000001


# Final comparison for you all dekhlo guys

In [78]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = []
predictions = {
    "Logistic Regression": y_pred,
    "KNN": y_pred_knn,
    "Naive Bayes": y_pred_nb,
    "Decision Tree": y_pred_DT,
    "SVM": y_pred_svc,
}

for name, preds in predictions.items():
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, preds), 4),
        "Precision": round(precision_score(y_test, preds, average='weighted'), 4),
        "Recall": round(recall_score(y_test, preds, average='weighted'), 4),
        "F1 Score": round(f1_score(y_test, preds, average='weighted'), 4),
    })

results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)
results_df

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.863,0.8634,0.863,0.8631
2,Naive Bayes,0.863,0.8634,0.863,0.8631
4,SVM,0.863,0.8634,0.863,0.8631
1,KNN,0.860,0.8606,0.860,0.8601
3,Decision Tree,0.681,0.7395,0.681,0.6717
